## 1. Data Cleaning

This cleaning is done to a get most clear data out of the [glassdoor_jobs.csv](../data/raw/glassdoor_jobs.csv)

In [23]:
import sys, re, os
import pandas as pd

##### Import custom functions

In [24]:
sys.path.append("..")
from src import title_simplifier, seniority

In [25]:
df = pd.read_csv('../data/raw/glassdoor_jobs.csv', index_col=0)
df.head()

,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors
0,Data Scientist,$53K-$91K (Glassdoor est.),"Data Scientist\nLocation: Albuquerque, NM\nEdu...",3.8,Tecolote Research\n3.8,"Albuquerque, NM","Goleta, CA",501 to 1000 employees,1973,Company - Private,Aerospace & Defense,Aerospace & Defense,$50 to $100 million (USD),-1
1,Healthcare Data Scientist,$63K-$112K (Glassdoor est.),What You Will Do:\n\nI. General Summary\n\nThe...,3.4,University of Maryland Medical System\n3.4,"Linthicum, MD","Baltimore, MD",10000+ employees,1984,Other Organization,Health Care Services & Hospitals,Health Care,$2 to $5 billion (USD),-1
2,Data Scientist,$80K-$90K (Glassdoor est.),"KnowBe4, Inc. is a high growth information sec...",4.8,KnowBe4\n4.8,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,2010,Company - Private,Security Services,Business Services,$100 to $500 million (USD),-1
3,Data Scientist,$56K-$97K (Glassdoor est.),*Organization and Job ID**\nJob ID: 310709\n\n...,3.8,PNNL\n3.8,"Richland, WA","Richland, WA",1001 to 5000 employees,1965,Government,Energy,"Oil, Gas, Energy & Utilities",$500 million to $1 billion (USD),"Oak Ridge National Laboratory, National Renewa..."
4,Data Scientist,$86K-$143K (Glassdoor est.),Data Scientist\nAffinity Solutions / Marketing...,2.9,Affinity Solutions\n2.9,"New York, NY","New York, NY",51 to 200 employees,1998,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable,"Commerce Signals, Cardlytics, Yodlee"


#### Structure of dataset

Should identify the data type, it structure and it's count

In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 956 entries, 0 to 955
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Job Title          956 non-null    str    
 1   Salary Estimate    956 non-null    str    
 2   Job Description    956 non-null    str    
 3   Rating             956 non-null    float64
 4   Company Name       956 non-null    str    
 5   Location           956 non-null    str    
 6   Headquarters       956 non-null    str    
 7   Size               956 non-null    str    
 8   Founded            956 non-null    int64  
 9   Type of ownership  956 non-null    str    
 10  Industry           956 non-null    str    
 11  Sector             956 non-null    str    
 12  Revenue            956 non-null    str    
 13  Competitors        956 non-null    str    
dtypes: float64(1), int64(1), str(12)
memory usage: 104.7 KB


In [27]:
print(f"Before modification data entry count: {len(df)}")

df = df.drop_duplicates()
print(f"After modification data entry count: {len(df)}")
df.head()

Before modification data entry count: 956
After modification data entry count: 600


,Job Title,Salary Estimate,Job Description,Rating,Company Name,Location,Headquarters,Size,Founded,Type of ownership,Industry,Sector,Revenue,Competitors
0,Data Scientist,$53K-$91K (Glassdoor est.),"Data Scientist\nLocation: Albuquerque, NM\nEdu...",3.8,Tecolote Research\n3.8,"Albuquerque, NM","Goleta, CA",501 to 1000 employees,1973,Company - Private,Aerospace & Defense,Aerospace & Defense,$50 to $100 million (USD),-1
1,Healthcare Data Scientist,$63K-$112K (Glassdoor est.),What You Will Do:\n\nI. General Summary\n\nThe...,3.4,University of Maryland Medical System\n3.4,"Linthicum, MD","Baltimore, MD",10000+ employees,1984,Other Organization,Health Care Services & Hospitals,Health Care,$2 to $5 billion (USD),-1
2,Data Scientist,$80K-$90K (Glassdoor est.),"KnowBe4, Inc. is a high growth information sec...",4.8,KnowBe4\n4.8,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,2010,Company - Private,Security Services,Business Services,$100 to $500 million (USD),-1
3,Data Scientist,$56K-$97K (Glassdoor est.),*Organization and Job ID**\nJob ID: 310709\n\n...,3.8,PNNL\n3.8,"Richland, WA","Richland, WA",1001 to 5000 employees,1965,Government,Energy,"Oil, Gas, Energy & Utilities",$500 million to $1 billion (USD),"Oak Ridge National Laboratory, National Renewa..."
4,Data Scientist,$86K-$143K (Glassdoor est.),Data Scientist\nAffinity Solutions / Marketing...,2.9,Affinity Solutions\n2.9,"New York, NY","New York, NY",51 to 200 employees,1998,Company - Private,Advertising & Marketing,Business Services,Unknown / Non-Applicable,"Commerce Signals, Cardlytics, Yodlee"


#### Clean the column names

In [28]:
print(f'Original Column Names:  {df.columns.tolist()}')

df.columns = df.columns.str.lower().str.replace(' ', '_')

print(f'New Column Names:  {df.columns.tolist()}')

Original Column Names:  ['Job Title', 'Salary Estimate', 'Job Description', 'Rating', 'Company Name', 'Location', 'Headquarters', 'Size', 'Founded', 'Type of ownership', 'Industry', 'Sector', 'Revenue', 'Competitors']
New Column Names:  ['job_title', 'salary_estimate', 'job_description', 'rating', 'company_name', 'location', 'headquarters', 'size', 'founded', 'type_of_ownership', 'industry', 'sector', 'revenue', 'competitors']


#### Cleaning salary column

We should remove all invalid entries

In [29]:
print(f"Before modification data entry count: {len(df)}")
df = df[df['salary_estimate'] != '-1']
print(f"After modification data entry count: {len(df)}")

Before modification data entry count: 600
After modification data entry count: 467


We divide the salary_estimate value into more detailed columns

#### Cleaning company name column

Need to remove ratings in the company name as there is a rating column already available in the dataset

In [30]:
df['company_text'] = df['company_name'].apply(lambda x: x.split('\n')[0])
df[['company_text']].head(10)

,company_text
0,Tecolote Research
1,University of Maryland Medical System
2,KnowBe4
3,PNNL
4,Affinity Solutions
5,CyrusOne
6,ClearOne Advantage
7,Logic20/20
8,Rochester Regional Health
9,<intent>


#### Handle values in size column

In [31]:
# Removes the negative values
print(f'Previous size column value count: \n{df['size'].value_counts()}')
df = df[df['size'] != '-1']

print(f'\nCurrent size column value counts after removing negative values: \n{df['size'].value_counts()}')

Previous size column value count: 
size
1001 to 5000 employees     93
501 to 1000 employees      80
10000+ employees           80
201 to 500 employees       77
51 to 200 employees        61
5001 to 10000 employees    46
1 to 50 employees          24
Unknown                     5
-1                          1
Name: count, dtype: int64

Current size column value counts after removing negative values: 
size
1001 to 5000 employees     93
501 to 1000 employees      80
10000+ employees           80
201 to 500 employees       77
51 to 200 employees        61
5001 to 10000 employees    46
1 to 50 employees          24
Unknown                     5
Name: count, dtype: int64


#### Creating a new col out of the state from Job Location column

In [32]:
df['job_state'] = df['location'].apply(lambda x: x.split(',')[1].replace('Los Angeles', 'LA'))

# df.job_state.unique()
df.job_state.value_counts()

job_state
CA    98
MA    59
NY    47
VA    30
MD    22
IL    22
PA    19
TX    17
WA    15
NJ    13
NC    11
FL     9
DC     9
OH     9
CO     7
IN     7
TN     7
AL     6
MO     6
AZ     6
UT     6
WI     5
MI     4
LA     4
KY     3
OR     3
GA     3
NE     3
IA     3
NM     2
CT     2
MN     2
DE     2
ID     2
RI     1
SC     1
KS     1
Name: count, dtype: int64

## 2. Feature Engineering

#### 1. Salary Parsing

In [33]:
df['hourly'] = df['salary_estimate'].apply(lambda x: 1 if 'per hour' in x.lower() else 0)
df['employer_provided_salary'] = df['salary_estimate'].apply(lambda x: 1 if 'Employer Provided Salary' in x.lower() else 0)

salary = df['salary_estimate'].apply(lambda x: x.split('(')[0])
minus_kd = salary.apply(lambda x: x.replace('$', '').replace('K', ''))
minus_hr = minus_kd.apply(lambda x: x.lower().replace('per hour', '').replace('employer provided salary:', ''))

df['min_salary'] = minus_hr.apply(lambda x: float(x.split('-')[0]))
df['max_salary'] = minus_hr.apply(lambda x: float(x.split('-')[1]))
df['avg_salary'] = df['min_salary'] + df['min_salary'] / 2.0

df[['min_salary', 'max_salary', 'avg_salary']].head(10)

print(f"Min Salary Dtype: {df['min_salary'].dtype}")
print(f"Max Salary Dtype: {df['max_salary'].dtype}")
print(f"Average Salary Dtype: {df['avg_salary'].dtype}")

Min Salary Dtype: float64
Max Salary Dtype: float64
Average Salary Dtype: float64


#### 2. Location Parsing - Same state

We can check if the job location is the same state as the headquaters location

In [34]:
df['same_state'] = df.apply(lambda x: 1 if x.location == x.headquarters else 0, axis=1)
df.head()

,job_title,salary_estimate,job_description,rating,company_name,location,headquarters,size,founded,type_of_ownership,...,revenue,competitors,company_text,job_state,hourly,employer_provided_salary,min_salary,max_salary,avg_salary,same_state
0,Data Scientist,$53K-$91K (Glassdoor est.),"Data Scientist\nLocation: Albuquerque, NM\nEdu...",3.8,Tecolote Research\n3.8,"Albuquerque, NM","Goleta, CA",501 to 1000 employees,1973,Company - Private,...,$50 to $100 million (USD),-1,Tecolote Research,NM,0,0,53.0,91.0,79.5,0
1,Healthcare Data Scientist,$63K-$112K (Glassdoor est.),What You Will Do:\n\nI. General Summary\n\nThe...,3.4,University of Maryland Medical System\n3.4,"Linthicum, MD","Baltimore, MD",10000+ employees,1984,Other Organization,...,$2 to $5 billion (USD),-1,University of Maryland Medical System,MD,0,0,63.0,112.0,94.5,0
2,Data Scientist,$80K-$90K (Glassdoor est.),"KnowBe4, Inc. is a high growth information sec...",4.8,KnowBe4\n4.8,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,2010,Company - Private,...,$100 to $500 million (USD),-1,KnowBe4,FL,0,0,80.0,90.0,120.0,1
3,Data Scientist,$56K-$97K (Glassdoor est.),*Organization and Job ID**\nJob ID: 310709\n\n...,3.8,PNNL\n3.8,"Richland, WA","Richland, WA",1001 to 5000 employees,1965,Government,...,$500 million to $1 billion (USD),"Oak Ridge National Laboratory, National Renewa...",PNNL,WA,0,0,56.0,97.0,84.0,1
4,Data Scientist,$86K-$143K (Glassdoor est.),Data Scientist\nAffinity Solutions / Marketing...,2.9,Affinity Solutions\n2.9,"New York, NY","New York, NY",51 to 200 employees,1998,Company - Private,...,Unknown / Non-Applicable,"Commerce Signals, Cardlytics, Yodlee",Affinity Solutions,NY,0,0,86.0,143.0,129.0,1


#### 3. Age of the company

We can check the comapany's age from the year column to now.

In [35]:
df['company_age'] = df.founded.apply(lambda x: x if x < 1 else 2026 - x)
df[['company_age']].head()

,company_age
0,53
1,42
2,16
3,61
4,28


#### 4. Parsing Company Size

We can create structured data out of comapny size which later helps in modeling

1. Min Employees
2. Max Employees
3. Average Employees

In [36]:
cleaned_company_size = (
    df['size']
    .str.lower()
    .str.replace('employees', '', regex=False)
    .str.replace('unknown', '', regex=False)
    .str.replace('+', '', regex=False)
    .str.strip()
)

df['min_emp_count'] = cleaned_company_size.str.split('to').str[0].str.strip()
df['max_emp_count'] = cleaned_company_size.str.split('to').str[1].str.strip()

df['min_emp_count'] = df['min_emp_count'].fillna(-1)
df['max_emp_count'] = (df['max_emp_count'].fillna(df['min_emp_count']) if df['min_emp_count'].any() != -1 else -1)

df['min_emp_count'] = pd.to_numeric(df['min_emp_count'], errors='coerce')
df['max_emp_count'] = pd.to_numeric(df['max_emp_count'], errors='coerce')

df['avg_emp_count'] = ((df['max_emp_count'] + df['min_emp_count']) / 2).round().astype('Int64')
df[['min_emp_count', 'max_emp_count', 'avg_emp_count']].value_counts()

min_emp_count  max_emp_count  avg_emp_count
1001.0         5000.0         3000             93
501.0          1000.0         750              80
10000.0        10000.0        10000            80
201.0          500.0          350              77
51.0           200.0          126              61
5001.0         10000.0        7500             46
1.0            50.0           26               24
Name: count, dtype: int64

#### 5. Creating required skills using job description

Most of the commmon skills required in data science is:

    1. Python
    2. R
    3. Power BI
    4. SQL
    5. Excel
    6. Spark

In [37]:
skill_set = ['python', 'r', 'power_bi', 'sql', 'excel', 'spark']

desc_lower = df['job_description'].str.lower()

for skill in skill_set:
    col_name = skill + "_yn"
    skill_name = r'\b' + re.escape(skill) + r'\b'
    df[col_name] = desc_lower.str.contains(skill_name, regex=True, na=False)

df.head()

,job_title,salary_estimate,job_description,rating,company_name,location,headquarters,size,founded,type_of_ownership,...,company_age,min_emp_count,max_emp_count,avg_emp_count,python_yn,r_yn,power_bi_yn,sql_yn,excel_yn,spark_yn
0,Data Scientist,$53K-$91K (Glassdoor est.),"Data Scientist\nLocation: Albuquerque, NM\nEdu...",3.8,Tecolote Research\n3.8,"Albuquerque, NM","Goleta, CA",501 to 1000 employees,1973,Company - Private,...,53,501.0,1000.0,750,True,False,False,False,True,False
1,Healthcare Data Scientist,$63K-$112K (Glassdoor est.),What You Will Do:\n\nI. General Summary\n\nThe...,3.4,University of Maryland Medical System\n3.4,"Linthicum, MD","Baltimore, MD",10000+ employees,1984,Other Organization,...,42,10000.0,10000.0,10000,True,True,False,False,False,False
2,Data Scientist,$80K-$90K (Glassdoor est.),"KnowBe4, Inc. is a high growth information sec...",4.8,KnowBe4\n4.8,"Clearwater, FL","Clearwater, FL",501 to 1000 employees,2010,Company - Private,...,16,501.0,1000.0,750,True,True,False,True,True,True
3,Data Scientist,$56K-$97K (Glassdoor est.),*Organization and Job ID**\nJob ID: 310709\n\n...,3.8,PNNL\n3.8,"Richland, WA","Richland, WA",1001 to 5000 employees,1965,Government,...,61,1001.0,5000.0,3000,True,False,False,False,False,False
4,Data Scientist,$86K-$143K (Glassdoor est.),Data Scientist\nAffinity Solutions / Marketing...,2.9,Affinity Solutions\n2.9,"New York, NY","New York, NY",51 to 200 employees,1998,Company - Private,...,28,51.0,200.0,126,True,True,False,True,False,False


#### 6. Simplifying Titles in Job Title

Need to properly categorize the job title and define common title as categories

In [38]:
df['title_simplified'] = df['job_title'].apply(title_simplifier)
df['title_simplified'].value_counts()

title_simplified
data scientist    192
na                 97
data engineer      75
analyst            71
manager            12
mle                11
director            8
Name: count, dtype: int64

#### 7. Simplifying Seniority in Job Title

This properly helps to identify the position level and it will later help in modeling (Ex. Senior Position earns this much, etc.)

In [39]:
df['seniority'] = df['job_title'].apply(seniority)
df['seniority'].value_counts()

seniority
na        339
senior    124
junior      3
Name: count, dtype: int64

#### 8. Identify the job description length

In [40]:
df['desc_len'] = df['job_description'].apply(lambda x: len(x))
df['desc_len'].value_counts()

desc_len
4203    3
3747    2
3490    2
3698    2
2327    2
       ..
4023    1
407     1
3911    1
3478    1
3813    1
Name: count, Length: 442, dtype: int64

#### 9. Competitor count

In [41]:
df['competitor_count'] = df['competitors'].apply(lambda x: len(x.split(',')) if x != '-1' else 0)
df['competitor_count'].value_counts()

competitor_count
0    284
3    149
2     24
1      8
4      1
Name: count, dtype: int64

#### 10. Hourly wage to annual

We need to calculate annual wage for the entries which only provide per hour wage

So approx we have to calculate wage for 2000 hours (40 hours/per week * 50 working weeks)

Since the wage is already scaled down from 1000 we need to just (per hour wage * 2)

In [42]:
df['min_salary'] = df.apply(lambda x: x.min_salary * 2 if x.hourly == -1 else x.min_salary, axis=1)
df['max_salary'] = df.apply(lambda x: x.max_salary * 2 if x.hourly == -1 else x.max_salary, axis=1)
df[df.hourly == 1][['hourly', 'min_salary', 'max_salary']]

,hourly,min_salary,max_salary
197,1,17.0,24.0
209,1,21.0,34.0
240,1,18.0,25.0
247,1,21.0,34.0
257,1,15.0,25.0
437,1,24.0,39.0
464,1,25.0,28.0
522,1,21.0,29.0
523,1,10.0,17.0
823,1,27.0,47.0


## 4. Exporting the cleaned dataset

Exporting the cleaned csv file to [data/cleaned](../data/cleaned) as [data/cleaned/salary_data_cleaned](../data/cleaned/salary_data_cleaned.csv)

In [43]:
os.makedirs('../data/cleaned', exist_ok=True) # Creates a new clean directory
df.to_csv('../data/cleaned/salary_data_cleaned.csv', index=False)

Confirming if its exported

In [44]:
if os.path.exists('../data/cleaned/salary_data_cleaned.csv'):
    print("Cleaned Dataset exists AS salary_data_cleaned.csv")
else: print("Cleaned Dataset exist")

Cleaned Dataset exists AS salary_data_cleaned.csv
